In [2]:
from ema_workbench.analysis.scenario_discovery_util import RuleInductionType
from ema_workbench.em_framework.salib_samplers import get_SALib_problem
from ema_workbench import (Model, RealParameter, TimeSeriesOutcome,
                           perform_experiments, ema_logging,Policy)

from ema_workbench import Samplers

from ema_workbench.analysis import feature_scoring
from ema_workbench.analysis.scenario_discovery_util import RuleInductionType
from SALib.analyze import sobol
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from openpyxl.styles.builtins import output
from ema_workbench import (
    Model,
    Policy,
    ema_logging,
    SequentialEvaluator,
    MultiprocessingEvaluator,
)
from dike_model_function import DikeNetwork  # @UnresolvedImport
from problem_formulation import get_model_for_problem_formulation, sum_over, sum_over_time
ema_logging.log_to_stderr(ema_logging.INFO)

# choose problem formulation number, between 0-5
# each problem formulation has its own list of outcomes
dike_model, planning_steps = get_model_for_problem_formulation(3)

In [3]:
# enlisting uncertainties, their types (RealParameter/IntegerParameter/CategoricalParameter), lower boundary, and upper boundary
import copy

for unc in dike_model.uncertainties:
    print(repr(unc))

uncertainties = copy.deepcopy(dike_model.uncertainties)


CategoricalParameter('discount rate 0', [0, 1, 2, 3])
CategoricalParameter('discount rate 1', [0, 1, 2, 3])
CategoricalParameter('discount rate 2', [0, 1, 2, 3])
IntegerParameter('A.0_ID flood wave shape', 0, 132, resolution=None, default=None, variable_name=['A.0_ID flood wave shape'], pff=False)
RealParameter('A.1_Bmax', 30, 350, resolution=None, default=None, variable_name=['A.1_Bmax'], pff=False)
RealParameter('A.1_pfail', 0, 1, resolution=None, default=None, variable_name=['A.1_pfail'], pff=False)
CategoricalParameter('A.1_Brate', [0, 1, 2])
RealParameter('A.2_Bmax', 30, 350, resolution=None, default=None, variable_name=['A.2_Bmax'], pff=False)
RealParameter('A.2_pfail', 0, 1, resolution=None, default=None, variable_name=['A.2_pfail'], pff=False)
CategoricalParameter('A.2_Brate', [0, 1, 2])
RealParameter('A.3_Bmax', 30, 350, resolution=None, default=None, variable_name=['A.3_Bmax'], pff=False)
RealParameter('A.3_pfail', 0, 1, resolution=None, default=None, variable_name=['A.3_pfai

In [4]:
problem = get_SALib_problem(uncertainties)
print(problem)

{'num_vars': 19, 'names': ['A.0_ID flood wave shape', 'A.1_Bmax', 'A.1_Brate', 'A.1_pfail', 'A.2_Bmax', 'A.2_Brate', 'A.2_pfail', 'A.3_Bmax', 'A.3_Brate', 'A.3_pfail', 'A.4_Bmax', 'A.4_Brate', 'A.4_pfail', 'A.5_Bmax', 'A.5_Brate', 'A.5_pfail', 'discount rate 0', 'discount rate 1', 'discount rate 2'], 'bounds': [(0, 133), (30, 350), (0, 3), (0, 1), (30, 350), (0, 3), (0, 1), (30, 350), (0, 3), (0, 1), (30, 350), (0, 3), (0, 1), (30, 350), (0, 3), (0, 1), (0, 4), (0, 4), (0, 4)]}


In [5]:
from ema_workbench import Policy

def get_do_nothing_dict():
    return {l.name: 0 for l in dike_model.levers}

policies = [

    # 🔹 Policy 360
    Policy(
        "policy_360",
        **dict(
            get_do_nothing_dict(),
            **{
                "1_RfR 2": 1, "2_RfR 1": 1, "2_RfR 2": 1, "3_RfR 0": 1, "3_RfR 2": 1,
                "4_RfR 1": 1, "4_RfR 2": 1,
                "A.1_DikeIncrease 0": 3, "A.1_DikeIncrease 1": 3, "A.1_DikeIncrease 2": 10,
                "A.2_DikeIncrease 0": 9, "A.2_DikeIncrease 1": 8, "A.2_DikeIncrease 2": 4,
                "A.3_DikeIncrease 0": 8, "A.3_DikeIncrease 1": 1, "A.3_DikeIncrease 2": 9,
                "A.4_DikeIncrease 1": 6, "A.4_DikeIncrease 2": 7,
                "A.5_DikeIncrease 0": 9, "A.5_DikeIncrease 1": 6, "A.5_DikeIncrease 2": 1,
                "EWS_DaysToThreat": 0
            }
        )
    ),

    # 🔹 Policy 946
    Policy(
        "policy_946",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 1": 1,
                "1_RfR 0": 1, "1_RfR 1": 1, "1_RfR 2": 1,
                "3_RfR 0": 1, "3_RfR 2": 1,
                "4_RfR 2": 1,
                "A.1_DikeIncrease 0": 1, "A.1_DikeIncrease 2": 10,
                "A.2_DikeIncrease 0": 5, "A.2_DikeIncrease 1": 8, "A.2_DikeIncrease 2": 8,
                "A.3_DikeIncrease 0": 1, "A.3_DikeIncrease 1": 10, "A.3_DikeIncrease 2": 2,
                "A.4_DikeIncrease 1": 8, "A.4_DikeIncrease 2": 3,
                "A.5_DikeIncrease 0": 10, "A.5_DikeIncrease 1": 7, "A.5_DikeIncrease 2": 8,
                "EWS_DaysToThreat": 1
            }
        )
    ),

    # 🔹 Policy 647
    Policy(
        "policy_647",
        **dict(
            get_do_nothing_dict(),
            **{
                "1_RfR 1": 1, "1_RfR 2": 1,
                "2_RfR 0": 1,
                "3_RfR 0": 1, "3_RfR 1": 1, "3_RfR 2": 1,
                "4_RfR 0": 1, "4_RfR 1": 1,
                "A.1_DikeIncrease 0": 3, "A.1_DikeIncrease 1": 4, "A.1_DikeIncrease 2": 6,
                "A.2_DikeIncrease 0": 8, "A.2_DikeIncrease 1": 5, "A.2_DikeIncrease 2": 5,
                "A.3_DikeIncrease 0": 2, "A.3_DikeIncrease 1": 10, "A.3_DikeIncrease 2": 10,
                "A.4_DikeIncrease 0": 4, "A.4_DikeIncrease 1": 6, "A.4_DikeIncrease 2": 8,
                "A.5_DikeIncrease 0": 10, "A.5_DikeIncrease 1": 1, "A.5_DikeIncrease 2": 2,
                "EWS_DaysToThreat": 3
            }
        )
    ),

    # 🔹 Policy 776
    Policy(
        "policy_776",
        **dict(
            get_do_nothing_dict(),
            **{
                "1_RfR 1": 1, "1_RfR 2": 1,
                "2_RfR 1": 1,
                "3_RfR 0": 1, "3_RfR 1": 1, "3_RfR 2": 1,
                "4_RfR 0": 1, "4_RfR 1": 1, "4_RfR 2": 1,
                "A.1_DikeIncrease 1": 9, "A.1_DikeIncrease 2": 9,
                "A.2_DikeIncrease 0": 3, "A.2_DikeIncrease 1": 6,
                "A.3_DikeIncrease 0": 8, "A.3_DikeIncrease 1": 4, "A.3_DikeIncrease 2": 1,
                "A.4_DikeIncrease 0": 1, "A.4_DikeIncrease 1": 10, "A.4_DikeIncrease 2": 2,
                "A.5_DikeIncrease 0": 8, "A.5_DikeIncrease 1": 6, "A.5_DikeIncrease 2": 2,
                "EWS_DaysToThreat": 2
            }
        )
    ),

    # 🔹 Policy 422
    Policy(
        "policy_422",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 1": 1,
                "1_RfR 0": 1, "1_RfR 1": 1, "1_RfR 2": 1,
                "2_RfR 0": 1, "2_RfR 1": 1, "2_RfR 2": 1,
                "3_RfR 0": 1, "3_RfR 1": 1, "3_RfR 2": 1,
                "4_RfR 0": 1, "4_RfR 2": 1,
                "A.1_DikeIncrease 0": 5, "A.1_DikeIncrease 1": 2,
                "A.2_DikeIncrease 0": 7, "A.2_DikeIncrease 1": 7, "A.2_DikeIncrease 2": 6,
                "A.3_DikeIncrease 0": 5, "A.3_DikeIncrease 1": 5,
                "A.4_DikeIncrease 0": 2, "A.4_DikeIncrease 1": 9,
                "A.5_DikeIncrease 0": 10, "A.5_DikeIncrease 2": 4,
                "EWS_DaysToThreat": 2
            }
        )
    ),
]



In [6]:
from ema_workbench import Policy

def get_do_nothing_dict():
    return {l.name: 0 for l in dike_model.levers}

policies += [  # append to existing policies list

    # 🔹 Policy 352
    Policy(
        "policy_352",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 0": 1, "0_RfR 2": 1,
                "1_RfR 1": 1, "1_RfR 2": 1,
                "3_RfR 0": 1,
                "4_RfR 0": 1,
                "A.1_DikeIncrease 0": 9, "A.1_DikeIncrease 1": 1, "A.1_DikeIncrease 2": 6,
                "A.2_DikeIncrease 0": 3, "A.2_DikeIncrease 1": 3, "A.2_DikeIncrease 2": 4,
                "A.3_DikeIncrease 0": 5, "A.3_DikeIncrease 1": 2, "A.3_DikeIncrease 2": 5,
                "A.4_DikeIncrease 0": 1, "A.4_DikeIncrease 1": 9, "A.4_DikeIncrease 2": 8,
                "A.5_DikeIncrease 1": 5, "A.5_DikeIncrease 2": 3,
                "EWS_DaysToThreat": 0
            }
        )
    ),

    # 🔹 Policy 14
    Policy(
        "policy_14",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 0": 1, "0_RfR 1": 1, "0_RfR 2": 1,
                "1_RfR 1": 1,
                "2_RfR 0": 1, "2_RfR 2": 1,
                "3_RfR 0": 1, "3_RfR 2": 1,
                "4_RfR 0": 1,
                "A.1_DikeIncrease 0": 4, "A.1_DikeIncrease 1": 5, "A.1_DikeIncrease 2": 1,
                "A.2_DikeIncrease 0": 2, "A.2_DikeIncrease 1": 2, "A.2_DikeIncrease 2": 2,
                "A.3_DikeIncrease 0": 1, "A.3_DikeIncrease 1": 6, "A.3_DikeIncrease 2": 1,
                "A.4_DikeIncrease 1": 5, "A.4_DikeIncrease 2": 5,
                "A.5_DikeIncrease 0": 1, "A.5_DikeIncrease 2": 7,
                "EWS_DaysToThreat": 0
            }
        )
    ),

    # 🔹 Policy 410
    Policy(
        "policy_410",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 0": 1,
                "1_RfR 0": 1, "1_RfR 1": 1, "1_RfR 2": 1,
                "2_RfR 2": 1,
                "3_RfR 0": 1,
                "4_RfR 1": 1,
                "A.1_DikeIncrease 0": 7, "A.1_DikeIncrease 1": 5,
                "A.2_DikeIncrease 1": 10, "A.2_DikeIncrease 2": 1,
                "A.3_DikeIncrease 0": 1, "A.3_DikeIncrease 1": 2, "A.3_DikeIncrease 2": 6,
                "A.4_DikeIncrease 0": 1, "A.4_DikeIncrease 1": 5,
                "A.5_DikeIncrease 1": 10, "A.5_DikeIncrease 2": 1,
                "EWS_DaysToThreat": 0
            }
        )
    ),

    # 🔹 Policy 736
    Policy(
        "policy_736",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 1": 1, "0_RfR 2": 1,
                "1_RfR 0": 1,
                "3_RfR 2": 1,
                "4_RfR 0": 1,
                "A.1_DikeIncrease 0": 6, "A.1_DikeIncrease 1": 4, "A.1_DikeIncrease 2": 1,
                "A.2_DikeIncrease 0": 6, "A.2_DikeIncrease 1": 3, "A.2_DikeIncrease 2": 2,
                "A.3_DikeIncrease 0": 10, "A.3_DikeIncrease 2": 8,
                "A.4_DikeIncrease 0": 7, "A.4_DikeIncrease 1": 10, "A.4_DikeIncrease 2": 9,
                "A.5_DikeIncrease 0": 1, "A.5_DikeIncrease 1": 1, "A.5_DikeIncrease 2": 7,
                "EWS_DaysToThreat": 1
            }
        )
    ),

    # 🔹 Policy 621
    Policy(
        "policy_621",
        **dict(
            get_do_nothing_dict(),
            **{
                "0_RfR 0": 1,
                "1_RfR 0": 1,
                "3_RfR 0": 1,
                "4_RfR 2": 1,
                "A.1_DikeIncrease 0": 8,
                "A.2_DikeIncrease 0": 10, "A.2_DikeIncrease 1": 5,
                "A.3_DikeIncrease 0": 4, "A.3_DikeIncrease 1": 7,
                "A.4_DikeIncrease 0": 1, "A.4_DikeIncrease 2": 2,
                "A.5_DikeIncrease 0": 1, "A.5_DikeIncrease 1": 2, "A.5_DikeIncrease 2": 10,
                "EWS_DaysToThreat": 0
            }
        )
    ),
]


In [7]:
policies

[Policy({'0_RfR 0': 0, '0_RfR 1': 0, '0_RfR 2': 0, '1_RfR 0': 0, '1_RfR 1': 0, '1_RfR 2': 1, '2_RfR 0': 0, '2_RfR 1': 1, '2_RfR 2': 1, '3_RfR 0': 1, '3_RfR 1': 0, '3_RfR 2': 1, '4_RfR 0': 0, '4_RfR 1': 1, '4_RfR 2': 1, 'EWS_DaysToThreat': 0, 'A.1_DikeIncrease 0': 3, 'A.1_DikeIncrease 1': 3, 'A.1_DikeIncrease 2': 10, 'A.2_DikeIncrease 0': 9, 'A.2_DikeIncrease 1': 8, 'A.2_DikeIncrease 2': 4, 'A.3_DikeIncrease 0': 8, 'A.3_DikeIncrease 1': 1, 'A.3_DikeIncrease 2': 9, 'A.4_DikeIncrease 0': 0, 'A.4_DikeIncrease 1': 6, 'A.4_DikeIncrease 2': 7, 'A.5_DikeIncrease 0': 9, 'A.5_DikeIncrease 1': 6, 'A.5_DikeIncrease 2': 1}),
 Policy({'0_RfR 0': 0, '0_RfR 1': 1, '0_RfR 2': 0, '1_RfR 0': 1, '1_RfR 1': 1, '1_RfR 2': 1, '2_RfR 0': 0, '2_RfR 1': 0, '2_RfR 2': 0, '3_RfR 0': 1, '3_RfR 1': 0, '3_RfR 2': 1, '4_RfR 0': 0, '4_RfR 1': 0, '4_RfR 2': 1, 'EWS_DaysToThreat': 1, 'A.1_DikeIncrease 0': 1, 'A.1_DikeIncrease 1': 0, 'A.1_DikeIncrease 2': 10, 'A.2_DikeIncrease 0': 5, 'A.2_DikeIncrease 1': 8, 'A.2_DikeInc